# 1. Inicialización del Entorno Spark

**Dimensión 4: Patrones espaciales y concentración geográfica** — Regresión por celdas/cuadrantes.

Integrante: **Cristhian Chuquitarqui Chura** — Data Geniuses

Procesamiento batch distribuido de las coordenadas de Citi Bike NYC.


In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("U1_Patrones_Espaciales_CitiBike")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "16")
    .getOrCreate()
)

print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/11 14:27:50 WARN Utils: Your hostname, Jhack, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/09/11 14:27:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/11 14:27:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.2.0


# 2. Definición del Esquema Explícito y Carga de Datos (Capa Bronze)

La ruta de datos es portable: en el contenedor usa `/opt/UNIDAD1/data/*.csv`;
en ejecución local `../data/*.csv`.


In [2]:
import os as _os

_has_opt = _os.path.exists("/opt/UNIDAD1")
_cwd = _os.getcwd()

DATA_PATH = "/opt/UNIDAD1/data/*.csv"
GOLD_ROOT = "/opt/UNIDAD1/gold"
if not _has_opt:
    DATA_PATH = _os.path.join(_cwd, "..", "data", "*.csv")
    GOLD_ROOT = _os.path.join(_cwd, "..", "gold")

from pyspark.sql.types import (
    StructType, StructField,
    StringType, TimestampType, DoubleType,
)

schema_citibike = StructType([
    StructField("ride_id", StringType(), True),
    StructField("rideable_type", StringType(), True),
    StructField("started_at", TimestampType(), True),
    StructField("ended_at", TimestampType(), True),
    StructField("start_station_name", StringType(), True),
    StructField("start_station_id", StringType(), True),
    StructField("end_station_name", StringType(), True),
    StructField("end_station_id", StringType(), True),
    StructField("start_lat", DoubleType(), True),
    StructField("start_lng", DoubleType(), True),
    StructField("end_lat", DoubleType(), True),
    StructField("end_lng", DoubleType(), True),
    StructField("member_casual", StringType(), True),
])

df = spark.read.schema(schema_citibike).option("header", True).csv(DATA_PATH)

26/09/11 14:27:54 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /home/jhackns/bigdata/Proyectos/grupo-x-citibike-intelligent-mobility/pyspark/jupyter/UNIDAD1/Dim-Espacial/../data/*.csv.
java.io.FileNotFoundException: File /home/jhackns/bigdata/Proyectos/grupo-x-citibike-intelligent-mobility/pyspark/jupyter/UNIDAD1/data/*.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catal

# 2.1 Verificación Inicial de Registros


In [3]:
total_registros = df.count()
print(f"Registros ingeridos: {total_registros:,}")

Registros ingeridos: 4,993,137


In [4]:
df.printSchema()

root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: timestamp (nullable = true)
 |-- ended_at: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_lat: double (nullable = true)
 |-- start_lng: double (nullable = true)
 |-- end_lat: double (nullable = true)
 |-- end_lng: double (nullable = true)
 |-- member_casual: string (nullable = true)



# 3. Limpieza Silver: Coordenadas Válidas

Retenemos viajes con latitud/longitud de origen y destino dentro de rangos
geográficos plausibles y duración operativa (1–180 min).


In [5]:
from pyspark.sql.functions import col, unix_timestamp

df = df.withColumn(
    "duration_minutes",
    (unix_timestamp(col("ended_at")) - unix_timestamp(col("started_at"))) / 60.0,
)

In [6]:
df_clean = df.filter(
    col("start_lat").between(40.0, 41.5)
    & col("start_lng").between(-74.5, -73.0)
    & col("end_lat").between(40.0, 41.5)
    & col("end_lng").between(-74.5, -73.0)
    & col("duration_minutes").between(1.0, 180.0)
    & col("start_lat").isNotNull()
    & col("start_lng").isNotNull()
)

n_validos = df_clean.count()
print(f"Viajes con coordenadas válidas: {n_validos:,} | Descartados: {total_registros - n_validos:,}")

Viajes con coordenadas válidas: 4,971,847 | Descartados: 21,290


In [7]:
df_clean.select(
    "ride_id", "start_lat", "start_lng", "end_lat", "end_lng",
    "started_at", "member_casual",
).show(5, truncate=False)

+----------------+------------------+------------------+-----------------+------------------+-----------------------+-------------+
|ride_id         |start_lat         |start_lng         |end_lat          |end_lng           |started_at             |member_casual|
+----------------+------------------+------------------+-----------------+------------------+-----------------------+-------------+
|FB2708B4615FC7B1|40.74056423633952 |-73.99852573871613|40.72170005235607|-74.00238141417502|2026-07-04 15:52:29.356|member       |
|6D9C1D2C900149A3|40.722103786686034|-73.99724900722504|40.72170005235607|-74.00238141417502|2026-07-09 00:31:21.213|member       |
|C945BA79D67B1EAE|40.804213         |-73.96699104      |40.8008363       |-73.9664492472    |2026-07-02 22:51:30.134|member       |
|9896AB3B453BED3B|40.722103786686034|-73.99724900722504|40.73401143      |-74.00293877      |2026-07-01 15:50:55.534|member       |
|ACADC9E7E131ECBC|40.74964          |-73.98805         |40.77632142182271|-7

# 4. Feature Engineering: Cuadrantes Geográficos Reproductibles

Redondeo de lat/lng de origen a **2 decimales** (≈ 1.1 km de lado) para formar
celdas geográficas y agregación del conteo de viajes **por celda y hora**.


In [8]:
from pyspark.sql.functions import (
    round as spark_round, hour, dayofweek, when,
    count, col as ccol, format_string,
)

df_celdas = (
    df_clean.withColumn("cell_lat", spark_round(ccol("start_lat"), 2))
            .withColumn("cell_lng", spark_round(ccol("start_lng"), 2))
            .withColumn(
                "cell_key",
                format_string("%.2f_%.2f", ccol("cell_lat"), ccol("cell_lng")),
            )
            .withColumn("start_hour", hour(ccol("started_at")))
            .withColumn("day_of_week", dayofweek(ccol("started_at")))
            .withColumn(
                "is_weekend",
                when(ccol("day_of_week").isin(1, 7), 1).otherwise(0),
            )
)

In [9]:
df_concentracion = (
    df_celdas.groupBy(
        "cell_key", "cell_lat", "cell_lng", "start_hour", "is_weekend",
    )
    .agg(count("*").alias("trip_count"))
)

print(f"Total de celdas/hora analizadas: {df_concentracion.count():,}")

Total de celdas/hora analizadas: 13,520


In [10]:
print("Muestra del dataset de concentración:")
df_concentracion.orderBy(ccol("trip_count").desc()).show(10, truncate=False)

Muestra del dataset de concentración:


+------------+--------+--------+----------+----------+----------+
|cell_key    |cell_lat|cell_lng|start_hour|is_weekend|trip_count|
+------------+--------+--------+----------+----------+----------+
|40.76_-73.98|40.76   |-73.98  |17        |0         |13786     |
|40.74_-73.99|40.74   |-73.99  |17        |0         |12924     |
|40.74_-73.99|40.74   |-73.99  |18        |0         |12841     |
|40.75_-73.99|40.75   |-73.99  |17        |0         |11574     |
|40.75_-73.99|40.75   |-73.99  |18        |0         |11372     |
|40.75_-73.98|40.75   |-73.98  |17        |0         |10823     |
|40.73_-73.99|40.73   |-73.99  |18        |0         |10062     |
|40.75_-74.00|40.75   |-74.0   |17        |0         |9696      |
|40.76_-73.97|40.76   |-73.97  |17        |0         |9522      |
|40.72_-73.99|40.72   |-73.99  |18        |0         |9146      |
+------------+--------+--------+----------+----------+----------+
only showing top 10 rows


In [11]:
print("Estadística descriptiva de trip_count por celda/hora:")
df_concentracion.describe("trip_count", "cell_lat", "cell_lng").show()

Estadística descriptiva de trip_count por celda/hora:


+-------+-----------------+-------------------+-------------------+
|summary|       trip_count|           cell_lat|           cell_lng|
+-------+-----------------+-------------------+-------------------+
|  count|            13520|              13520|              13520|
|   mean|367.7401627218935| 40.745573224852194| -73.94102440828453|
| stddev|861.1489490234233|0.06989385584017506|0.04588158242029026|
|    min|                1|              40.61|             -74.04|
|    max|            13786|              40.91|             -73.83|
+-------+-----------------+-------------------+-------------------+



# 5. Preparación ML: VectorAssembler y Split

Features geográficas y temporales empaquetadas con `VectorAssembler`;
etiqueta = `trip_count` (volumen de concentración por celda y hora).


In [12]:
from pyspark.ml.feature import VectorAssembler

feature_cols = ["cell_lat", "cell_lng", "start_hour", "is_weekend"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_model = assembler.transform(df_concentracion)
df_model = df_model.withColumn("label", ccol("trip_count").cast("double"))

train, test = df_model.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {train.count():,} | Test: {test.count():,}")

Train: 10,867 | Test: 2,653


# 6. Entrenamiento del Modelo Espacial (RandomForestRegressor)

Regresor distribuido no lineal para capturar la concentración geográfica.


In [13]:
from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(featuresCol="features", labelCol="label",
                           numTrees=50, maxDepth=10, seed=42)
model = rf.fit(train)
print("Modelo espacial entrenado correctamente.")

26/09/11 14:28:23 WARN DAGScheduler: Broadcasting large task binary with size 1020.7 KiB


26/09/11 14:28:23 WARN DAGScheduler: Broadcasting large task binary with size 1866.4 KiB


26/09/11 14:28:24 WARN DAGScheduler: Broadcasting large task binary with size 3.2 MiB


26/09/11 14:28:24 WARN DAGScheduler: Broadcasting large task binary with size 5.5 MiB


26/09/11 14:28:25 WARN DAGScheduler: Broadcasting large task binary with size 1408.0 KiB


Modelo espacial entrenado correctamente.


# 7. Muestra de Predicciones sobre el Conjunto de Test

Aplicamos el modelo y comparamos `trip_count` real frente a `prediction`.


In [14]:
predicciones = model.transform(test)
predicciones.select("cell_key", "trip_count", "prediction").show(10, truncate=False)

+------------+----------+------------------+
|cell_key    |trip_count|prediction        |
+------------+----------+------------------+
|40.61_-74.03|13        |27.641083912363197|
|40.61_-74.03|12        |13.761535699366991|
|40.61_-74.03|6         |30.8472818434598  |
|40.61_-74.04|7         |5.530976204250923 |
|40.62_-74.02|5         |13.743699123977715|
|40.62_-74.02|12        |16.715221930265912|
|40.62_-74.02|16        |68.61932890739949 |
|40.62_-74.03|9         |10.562293033276102|
|40.62_-74.04|1         |4.4686946628903605|
|40.62_-74.04|5         |9.101215740179114 |
+------------+----------+------------------+
only showing top 10 rows


# 8. Evaluación Formal del Modelo (Métricas de Calidad)

`RegressionEvaluator`: **RMSE**, **MAE** y **R²** sobre el test.


In [15]:
from pyspark.ml.evaluation import RegressionEvaluator

rmse = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse").evaluate(predicciones)
mae = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="mae").evaluate(predicciones)
r2 = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2").evaluate(predicciones)

In [16]:
print("=" * 52)
print("MÉTRICAS ESPACIALES (CONCENTRACIÓN POR CELDA/HORA)")
print("=" * 52)
print(f"  RMSE : {rmse:10.4f}")
print(f"  MAE  : {mae:10.4f}")
print(f"  R2   : {r2:10.4f}")
print("=" * 52)

MÉTRICAS ESPACIALES (CONCENTRACIÓN POR CELDA/HORA)
  RMSE :   345.0841
  MAE  :   135.5580
  R2   :     0.8281


# 9. Persistencia Analítica en Capa Gold

Guardamos el artefacto de concentración espacial y predicciones en **Parquet**.


In [17]:
from pyspark.sql.functions import round as spark_round

pred_final = (
    predicciones.select(
        "cell_key", "cell_lat", "cell_lng", "start_hour", "is_weekend",
        ccol("trip_count"),
        spark_round(ccol("prediction"), 2).alias("prediction"),
    )
)

GOLD_PATH = GOLD_ROOT + "/espacial_predictions.parquet"
pred_final.write.mode("overwrite").parquet(GOLD_PATH)
print(f"Artefacto persistido: {GOLD_PATH}")

Artefacto persistido: /home/jhackns/bigdata/Proyectos/grupo-x-citibike-intelligent-mobility/pyspark/jupyter/UNIDAD1/Dim-Espacial/../gold/espacial_predictions.parquet


In [18]:
gold_check = spark.read.parquet(GOLD_PATH)
print(f"Filas en capa Gold: {gold_check.count():,}")
gold_check.show(5, truncate=False)

Filas en capa Gold: 2,653
+------------+--------+--------+----------+----------+----------+----------+
|cell_key    |cell_lat|cell_lng|start_hour|is_weekend|trip_count|prediction|
+------------+--------+--------+----------+----------+----------+----------+
|40.61_-74.03|40.61   |-74.03  |6         |0         |3         |61.17     |
|40.61_-74.03|40.61   |-74.03  |10        |1         |3         |11.63     |
|40.61_-74.03|40.61   |-74.03  |14        |0         |17        |37.13     |
|40.61_-74.03|40.61   |-74.03  |15        |0         |11        |38.73     |
|40.61_-74.03|40.61   |-74.03  |16        |1         |9         |14.26     |
+------------+--------+--------+----------+----------+----------+----------+
only showing top 5 rows
